# 00 - Load Data

In [4]:
import pandas as pd
import numpy as np

## Cargar datos crudos

In [5]:
online_retail_ii = pd.read_excel(r"../data/raw/online_retail_ii.xlsx")

online_retail_ii.rename(columns={
    "Price" : "UnitPrice",
    "Customer ID" : "CustomerID"
}, inplace=True)

print(online_retail_ii.shape)
online_retail_ii.head(5)

(525461, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
online_retail = pd.read_excel(r"../data/raw/online_retail.xlsx")

online_retail.rename(columns={
    "InvoiceNo" : "Invoice"
}, inplace=True)

print(online_retail.shape)
online_retail.head(5)

(541909, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Combinar y limpiar

In [7]:
# Se omiten las ordenes de compra que ya estan en el primer dataset
online_retail = online_retail[~online_retail["Invoice"].isin(online_retail_ii["Invoice"].unique())]

# Union de datasets
df_raw = pd.concat([
    online_retail_ii,
    online_retail
])
del online_retail_ii, online_retail

# Ordenamiento de dataset
df_raw.sort_values(by=["InvoiceDate","Invoice","StockCode"], inplace=True)

cols_str = ["Invoice","StockCode","Description","Country"]
for col in cols_str:
    df_raw[col] = df_raw[col].astype(str)

cols_float = ["Quantity","UnitPrice","CustomerID"]
for col in cols_float:
    df_raw[col] = df_raw[col].astype(float)

df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"])

print(df_raw.shape)
df_raw.head()

(1044847, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10.0,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24.0,2009-12-01 07:45:00,1.65,13085.0,United Kingdom


In [8]:
df = df_raw.copy()
del df_raw

df.dropna(subset="CustomerID", inplace=True)
df.sort_values(by=["CustomerID","InvoiceDate","Invoice","StockCode"], inplace=True)
df.reset_index(drop=True, inplace=True)

df["Sales"] = df["Quantity"] * df["UnitPrice"]

print(df.shape)
df.head()

(809560, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,491725,TEST001,This is a test product.,10.0,2009-12-14 08:34:00,4.5,12346.0,United Kingdom,45.0
1,491742,TEST001,This is a test product.,5.0,2009-12-14 11:00:00,4.5,12346.0,United Kingdom,22.5
2,491744,TEST001,This is a test product.,5.0,2009-12-14 11:02:00,4.5,12346.0,United Kingdom,22.5
3,492718,TEST001,This is a test product.,5.0,2009-12-18 10:47:00,4.5,12346.0,United Kingdom,22.5
4,492722,TEST002,This is a test product.,1.0,2009-12-18 10:55:00,1.0,12346.0,United Kingdom,1.0


## Construir dataset a nivel cliente

In [9]:
df_prod = df.copy()

clientes_de_pruebas = df_prod[df_prod["StockCode"].str.casefold().str.contains('test')]["CustomerID"].drop_duplicates()
df_prod = df_prod[~df_prod["CustomerID"].isin(clientes_de_pruebas)]

df_prod_compras = df_prod[df_prod["Quantity"]>=0]

df_prod_devolucion = df_prod[df_prod["Quantity"]<0]

# -- Comrpas reales --
df_customer = df_prod_compras.groupby(by=["CustomerID"]).agg({
    "Country" : pd.Series.nunique,
    "InvoiceDate" : ["min","max"],
    "Sales" : "sum",
    "Quantity" : "sum",
    "Invoice" : pd.Series.nunique,
    "UnitPrice" : "max",
    "StockCode" : pd.Series.nunique
})
df_customer.columns = ["Paises Distintos","Date_Min","Date_Max",  "Sales","Quantity","Compras","Precio_Max","Productos Distintos"]
df_customer.reset_index(inplace=True)


df_customer_top_country = df_prod_compras.groupby(by=["CustomerID","Country"])["Invoice"].nunique().rename("Compras").reset_index().sort_values(by=["CustomerID","Compras"], ascending=False)

df_customer_top_country["rn"] = df_customer_top_country.groupby(by="CustomerID")["CustomerID"].cumcount()
df_customer_top_country = df_customer_top_country[df_customer_top_country["rn"]==0].drop(columns=["rn","Compras"]).rename(columns={"Country":"Pais Principal"})

df_customer = pd.merge(
    df_customer,
    df_customer_top_country,
    on="CustomerID",
    how="left"
)

df_customer.set_index("CustomerID", inplace=True)

df_customer["Permanencia"] = np.round(( df_customer["Date_Max"] - df_customer["Date_Min"] ).dt.total_seconds()/60/60/24, 0) + 1

df_customer["Canasta_Prom"] = df_customer["Quantity"]/df_customer["Compras"]
df_customer["Ticket_Prom"] = df_customer["Sales"]/df_customer["Compras"]
df_customer["Precio_Prom"] = df_customer["Sales"]/df_customer["Quantity"]


# -- Devoluciones --

df_customer_devolucion = df_prod_devolucion.groupby(by="CustomerID")["Invoice"].nunique().rename("Devoluciones").reset_index()


# -- Union de datos --

df_customer = pd.merge(
    df_customer,
    df_customer_devolucion,
    on="CustomerID",
    how="left"
)
df_customer.loc[df_customer["Devoluciones"].isna(), "Devoluciones"] = 0

df_customer["Pct_Devoluciones"] = df_customer["Devoluciones"] / df_customer["Compras"]

df_customer["Pais Principal"] = df_customer["Pais Principal"].astype("category")
df_customer["Paises Distintos"] = df_customer["Paises Distintos"].astype("category")

df_customer = df_customer.set_index("CustomerID")[[
    "Pais Principal", "Paises Distintos",
    "Permanencia",
    "Compras",
    "Canasta_Prom", "Ticket_Prom", "Precio_Prom", "Precio_Max",
    "Productos Distintos",
    "Pct_Devoluciones"
]]

print(df_customer.shape)
df_customer.head()

(5876, 10)


,Pais Principal,Paises Distintos,Permanencia,Compras,Canasta_Prom,Ticket_Prom,Precio_Prom,Precio_Max,Productos Distintos,Pct_Devoluciones
CustomerID,,,,,,,,,,
12347.0,Iceland,1,403.0,8,370.875,615.19125,1.658756,12.75,126,0.00
12348.0,Finland,1,364.0,5,542.800,403.88000,0.744068,40.00,25,0.00
12349.0,Italy,1,572.0,4,406.000,1107.17250,2.727026,300.00,138,0.25
12350.0,Norway,1,1.0,1,197.000,334.40000,1.697462,40.00,17,0.00
12351.0,Unspecified,1,1.0,1,261.000,300.93000,1.152989,12.75,21,0.00


## Guardar dataset procesado

In [10]:
df_customer.to_csv("../data/processed/customer_features.csv", encoding="utf-8")